# Stage 04 — Cut Clips hotfix

Notebook dùng chung cho Tier 1/2/3. Logic nằm trong `cut_clips_core.py` tại một Git SHA cố định. Không sửa code trực tiếp trên Kaggle và không tái sử dụng `run_id`.

In [ ]:
# ===== Chỉ sửa cell cấu hình này =====
TIER = "tier1"                         # tier1 | tier2 | tier3
HOTFIX_GIT_SHA = "__SET_AFTER_PUSH__" # commit chứa hotfix, không dùng main trôi nổi
RUN_ID = "__SET_RUN_ID__"             # vd cutfix_20260729_ab12cd3
START_INDEX = 0                        # inclusive
END_INDEX = 100                        # exclusive
NUM_WORKERS = 4

# Manifest quality-pass lấy từ repo đã checkout đúng SHA.
# Tier 2/3 phải điền đúng Kaggle mount của raw media.
DATASET_DIR_OVERRIDE = ""
INPUT_CSV_OVERRIDE = ""


In [ ]:
import os, pathlib, subprocess, sys

if HOTFIX_GIT_SHA.startswith("__") or RUN_ID.startswith("__"):
    raise ValueError("Phải khóa HOTFIX_GIT_SHA và RUN_ID trước khi chạy")
if TIER not in {"tier1", "tier2", "tier3"}:
    raise ValueError(f"Tier không hợp lệ: {TIER}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ultralytics", "opencv-python", "tqdm"], check=True)
repo = pathlib.Path("/kaggle/working/VN-AV-DF-Capstone")
if not repo.exists():
    subprocess.run(["git", "clone", "--filter=blob:none",
                    "https://github.com/nhatm2400/VN-AV-DF-Capstone.git",
                    str(repo)], check=True)
subprocess.run(["git", "fetch", "origin", HOTFIX_GIT_SHA], cwd=repo, check=True)
subprocess.run(["git", "checkout", "--detach", HOTFIX_GIT_SHA], cwd=repo, check=True)
actual_sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo,
                            capture_output=True, text=True, check=True).stdout.strip()
if actual_sha != HOTFIX_GIT_SHA:
    raise RuntimeError(f"Checkout sai SHA: {actual_sha} != {HOTFIX_GIT_SHA}")

face_model = pathlib.Path("/kaggle/working/yolov8n-face.pt")
if not face_model.exists():
    subprocess.run(["wget", "-q", "-O", str(face_model),
                    "https://huggingface.co/junjiang/GestureFace/resolve/main/yolov8n-face.pt"],
                   check=True)
print("Locked Git SHA:", actual_sha)


In [ ]:
import json

core_dir = repo / "src/pipeline/01_collect"
sys.path.insert(0, str(core_dir))
from cut_clips_core import CutConfig, run_batch

config_path = core_dir / "configs" / f"{TIER}.json"
config = CutConfig.from_json(config_path)
repo_manifest = repo / "data/01_collect" / f"{TIER}_quality_gate_passed.csv"
if not repo_manifest.is_file():
    raise FileNotFoundError(f"Thiếu manifest đã khóa trong Git: {repo_manifest}")
config.input_csv = str(repo_manifest)
config.run_id = RUN_ID
config.start_index = START_INDEX
config.end_index = END_INDEX
config.num_workers = NUM_WORKERS
config.face_model = str(face_model)
if DATASET_DIR_OVERRIDE:
    config.dataset_dir = DATASET_DIR_OVERRIDE
if INPUT_CSV_OVERRIDE:
    config.input_csv = INPUT_CSV_OVERRIDE

print(json.dumps(config.__dict__, ensure_ascii=False, indent=2))
config.validate()


In [ ]:
# Job dài chỉ bắt đầu ở cell này. Chạy sau khi đã kiểm tra config ở cell trên.
batch_dir = run_batch(config)
print("DONE:", batch_dir)


In [ ]:
# Gate nhanh sau run; run_batch đã fail trước đó nếu coverage/media contract sai.
summary = json.loads((batch_dir / "run_summary.json").read_text(encoding="utf-8"))
assert summary["coverage_passed"] is True
assert summary["batch_inputs"] == summary["terminal_statuses"]
summary
